# P2 CHAIN D — answers (archive only)

Marksheet-oriented reference. Open only when marking.


## Answer — D1


In [ ]:
# Answer — D1
from flask import Flask, render_template, request, redirect, url_for
import sqlite3

app = Flask(__name__)

def get_db():
    conn = sqlite3.connect("chain_a.db")
    conn.row_factory = sqlite3.Row
    return conn

@app.route("/", methods=["GET", "POST"])
def front():
    error = None
    if request.method == "POST":
        q_session = request.form.get("q_session", "").strip()
        q_name = request.form.get("q_name", "").strip()
        if q_session and q_name:
            return redirect(url_for("show", q_session=q_session, q_name=q_name))
        error = "Please fill in all fields"
    return render_template("chain_d/front.html", error=error)

@app.route("/show/<q_session>/<q_name>")
def show(q_session, q_name):
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT Member.name, Attendance.date, Attendance.session "
        "FROM Member INNER JOIN Attendance ON Member.member_id = Attendance.member_id "
        "WHERE Attendance.session = ? AND Member.name LIKE ? "
        "ORDER BY Attendance.date",
        (q_session, f"%{q_name}%"),
    )
    rows = cur.fetchall()
    conn.close()
    return render_template(
        "chain_d/show.html", rows=rows, q_session=q_session, q_name=q_name
    )

@app.route("/record", methods=["GET", "POST"])
def record():
    error = None
    if request.method == "POST":
        in_mid = request.form.get("in_mid", "").strip()
        in_date = request.form.get("in_date", "").strip()
        in_session = request.form.get("in_session", "").strip()
        if not (in_mid and in_date and in_session):
            error = "Please fill in all fields"
        else:
            conn = get_db()
            cur = conn.cursor()
            cur.execute(
                "INSERT OR IGNORE INTO Attendance(date, member_id, session) VALUES (?,?,?)",
                (in_date, in_mid, in_session),
            )
            conn.commit()
            conn.close()
            return redirect(url_for("front"))
    return render_template("chain_d/record.html", error=error)

if __name__ == "__main__":
    app.run(port=5003)


## Answer — D2


In [ ]:
# Answer — D2
class Node:
    def __init__(self, data=None):
        self.__data = data
        self.__right = None

    def get_data(self):
        return self.__data

    def set_data(self, data):
        self.__data = data

    def get_right(self):
        return self.__right

    def set_right(self, right):
        self.__right = right

class LinkedList:
    def __init__(self):
        self.__start = None

    def is_empty(self):
        return self.__start is None

    def insert_end(self, data):
        new = Node()
        new.set_data(data)
        if self.is_empty():
            self.__start = new
        else:
            current = self.__start
            while current.get_right() is not None:
                current = current.get_right()
            current.set_right(new)

    def search(self, data):
        current = self.__start
        while current is not None and current.get_data() != data:
            current = current.get_right()
        return current

    def delete(self, data):
        if self.is_empty():
            return
        if self.__start.get_data() == data:
            self.__start = self.__start.get_right()
            return
        previous = None
        current = self.__start
        while current is not None and current.get_data() != data:
            previous = current
            current = current.get_right()
        if current is None:
            return
        previous.set_right(current.get_right())

    def walk_ids(self):
        out = []
        current = self.__start
        while current is not None:
            out.append(current.get_data())
            current = current.get_right()
        return out

ids = [10, 20, 30, 40, 50]
catalogue = LinkedList()
for x in ids:
    catalogue.insert_end(x)
catalogue.delete(10)
print(catalogue.search(30) is not None, catalogue.search(-1), catalogue.walk_ids())
catalogue.delete(-1)


## Answer — D3


In [ ]:
# Answer — D3
class HashTable:
    def __init__(self, N):
        self.__n = N
        self.__arr = [LinkedList() for _ in range(N)]

    def hash_function(self, key):
        return key % self.__n

    def insert(self, key):
        bucket = self.__arr[self.hash_function(key)]
        if bucket.search(key) is None:
            bucket.insert_end(key)

    def search(self, key):
        return self.__arr[self.hash_function(key)].search(key) is not None

class TreeNode:
    def __init__(self, data):
        self.__data = data
        self.__left = None
        self.__right = None

    def get_data(self):
        return self.__data

    def get_left(self):
        return self.__left

    def set_left(self, node):
        self.__left = node

    def get_right(self):
        return self.__right

    def set_right(self, node):
        self.__right = node

class BST:
    def __init__(self):
        self.__root = None

    def get_root(self):
        return self.__root

    def insert(self, data):
        new = TreeNode(data)
        if self.__root is None:
            self.__root = new
            return
        current = self.__root
        while current:
            if data <= current.get_data():
                if current.get_left() is None:
                    current.set_left(new)
                    return
                current = current.get_left()
            else:
                if current.get_right() is None:
                    current.set_right(new)
                    return
                current = current.get_right()

    def in_order(self, node):
        if node is not None:
            self.in_order(node.get_left())
            print(node.get_data())
            self.in_order(node.get_right())

    def pre_order(self, node):
        if node is not None:
            print(node.get_data())
            self.pre_order(node.get_left())
            self.pre_order(node.get_right())

    def post_order(self, node):
        if node is not None:
            self.post_order(node.get_left())
            self.post_order(node.get_right())
            print(node.get_data())

ht = HashTable(7)
bst = BST()
for rid in catalogue.walk_ids():
    ht.insert(rid)
    bst.insert(rid)
print(ht.search(30))
bst.in_order(bst.get_root())


## Answer — D4


In [ ]:
# Answer — D4
from flask import Flask, render_template, request, redirect, url_for
from pymongo import MongoClient

app_mongo = Flask("chain_d_mongo")

def get_mongo():
    return MongoClient("mongodb://localhost:27017/")["chain_d_db"]["staff"]

# optional seed
_coll = get_mongo()
_coll.drop()
_coll.insert_many([
    {"staff_id": 1, "surname": "Tan", "given": "Mei", "department": "Maths"},
    {"staff_id": 2, "surname": "Lim", "given": "Wei", "department": "Computing"},
    {"staff_id": 3, "surname": "Ong", "given": "Siti", "department": "Science"},
])

@app_mongo.route("/", methods=["GET", "POST"])
def lobby():
    error = None
    if request.method == "POST":
        f_dept = request.form.get("f_dept", "").strip()
        f_surname = request.form.get("f_surname", "").strip()
        if f_dept and f_surname:
            return redirect(url_for("listed", f_dept=f_dept, f_surname=f_surname))
        error = "Please fill in all fields"
    return render_template("chain_d/lobby.html", error=error)

@app_mongo.route("/listed/<f_dept>/<f_surname>")
def listed(f_dept, f_surname):
    people = list(get_mongo().find(
        {"department": {"$regex": f_dept}, "surname": {"$regex": f_surname}},
        {"_id": 0},
    ))
    return render_template("chain_d/listed.html", people=people)

@app_mongo.route("/onboard", methods=["GET", "POST"])
def onboard():
    error = None
    if request.method == "POST":
        f_sid = request.form.get("f_sid", "").strip()
        f_surname = request.form.get("f_surname", "").strip()
        f_given = request.form.get("f_given", "").strip()
        f_dept = request.form.get("f_dept", "").strip()
        if not (f_sid and f_surname and f_given and f_dept):
            error = "Please fill in all fields"
        else:
            get_mongo().insert_one({
                "staff_id": int(f_sid),
                "surname": f_surname,
                "given": f_given,
                "department": f_dept,
            })
            return redirect(url_for("lobby"))
    return render_template("chain_d/onboard.html", error=error)

if __name__ == "__main__":
    app_mongo.run(port=5004)


## Answer — D5 (last sockets)


In [ ]:
# Answer — D5
def recv_line(sock):
    data = b""
    while b"\n" not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode("utf-8").strip()
# server: socket → bind → listen → accept → recv_line / sendall → close
# client: socket → connect → sendall((line+"\n").encode()) → recv → close
